# LLM Fine-Tuning & Alignment for AI Engineers

Fine-tune LLMs with LoRA/QLoRA, align models with DPO, and generate synthetic training data at scale.

**Topics:** Instruction dataset creation, LoRA/QLoRA with PEFT, DPO, RLHF overview, synthetic data generation, model merging

## 1. When to Fine-Tune vs Prompt Engineer

In [ ]:
from dataclasses import dataclass

@dataclass
class FineTuneDecision:
    scenario: str
    recommendation: str
    rationale: str

decisions = [
    FineTuneDecision(
        'Classify support tickets into 5 categories',
        'Few-shot prompting',
        'Simple task; 5 examples in prompt achieves >95% accuracy',
    ),
    FineTuneDecision(
        'Generate SQL from natural language (company schema)',
        'Fine-tune OR RAG with schema',
        'Schema is too large for context; need domain adaptation',
    ),
    FineTuneDecision(
        'Match company writing style in emails',
        'Fine-tune (LoRA)',
        'Style is hard to describe in prompt; 500 examples captures it well',
    ),
    FineTuneDecision(
        'Answer medical questions with citations',
        'RAG + prompting',
        'Facts change; model needs access to current literature',
    ),
    FineTuneDecision(
        'Reduce latency/cost by 10x for simple tasks',
        'Fine-tune small model (1-3B)',
        'Distill GPT-4 behavior into Phi-3-mini with LoRA',
    ),
]

print('Fine-tuning decision guide:')
print()
for d in decisions:
    print(f'Scenario: {d.scenario}')
    print(f'  → {d.recommendation}')
    print(f'  Why: {d.rationale}')
    print()

print('Rule of thumb: Fine-tune when you need consistent FORMAT or STYLE.')
print('Use RAG when you need up-to-date or proprietary FACTS.')

## 2. Instruction Dataset Creation — Self-Instruct with GPT-4

In [ ]:
import json
import os
from openai import OpenAI
from typing import Optional

client = OpenAI(api_key=os.getenv('OPENAI_API_KEY', 'sk-...'))

SELF_INSTRUCT_PROMPT = """You are generating training data for fine-tuning a language model.

Domain: {domain}
Task type: {task_type}

Generate a diverse, high-quality instruction-response pair.
The instruction should be realistic — something a real user would ask.
The response should be expert-level, accurate, and appropriately detailed.

Return JSON with keys: instruction, input (optional context), output

Example seed: {seed_example}"""

def generate_training_example(
    domain: str,
    task_type: str,
    seed_example: dict,
    model: str = 'gpt-4o-mini',
) -> Optional[dict]:
    """Generate one training example using self-instruct technique."""
    try:
        response = client.chat.completions.create(
            model=model,
            messages=[{'role': 'user', 'content': SELF_INSTRUCT_PROMPT.format(
                domain=domain, task_type=task_type,
                seed_example=json.dumps(seed_example),
            )}],
            response_format={'type': 'json_object'},
            temperature=0.9,  # high creativity for diversity
        )
        return json.loads(response.choices[0].message.content)
    except Exception:
        return None

# Example training data structure (Alpaca format)
sample_examples = [
    {'instruction': 'Explain gradient descent in simple terms.', 'input': '', 'output': 'Gradient descent is like walking downhill blindfolded...'},
    {'instruction': 'Write a Python function to compute BLEU score.', 'input': '', 'output': 'def bleu_score(reference, hypothesis):\n    ...'},
    {'instruction': 'What is the difference between precision and recall?', 'input': 'In the context of information retrieval.', 'output': 'Precision measures...'},
]

print('Training data format (Alpaca/Stanford format):')
for ex in sample_examples:
    print(json.dumps(ex, indent=2))
    print()
print(f'Minimum dataset size for fine-tuning: 500-1000 examples')
print(f'Self-instruct can generate 1000 examples for ~$2 with gpt-4o-mini')

## 3. LoRA Configuration and Training Setup

In [ ]:
from peft import LoraConfig, get_peft_model, TaskType
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments
from trl import SFTTrainer

def setup_lora_training(
    base_model: str = 'microsoft/Phi-3-mini-4k-instruct',
    lora_r: int = 16,
    lora_alpha: int = 32,
    lora_dropout: float = 0.05,
):
    """Configure LoRA — Low-Rank Adaptation of LLMs.
    
    LoRA freezes the original weights and adds trainable low-rank decomposition matrices:
    ΔW = A × B  where A is (d × r) and B is (r × d), r << d
    
    Only A and B are trained — typically 1-10% of original parameters.
    """
    lora_config = LoraConfig(
        r=lora_r,                          # rank: smaller = fewer params, larger = more capacity
        lora_alpha=lora_alpha,             # scaling: lora_alpha/r determines actual learning rate
        lora_dropout=lora_dropout,
        target_modules=['q_proj', 'v_proj', 'k_proj', 'o_proj'],  # which layers to adapt
        task_type=TaskType.CAUSAL_LM,
        bias='none',                       # don't train bias terms
    )
    
    training_args = TrainingArguments(
        output_dir='./lora-output',
        num_train_epochs=3,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=8,     # effective batch size = 16
        learning_rate=2e-4,
        lr_scheduler_type='cosine',
        warmup_ratio=0.1,
        logging_steps=10,
        save_strategy='epoch',
        bf16=True,
        report_to='mlflow',
    )
    
    return lora_config, training_args

# LoRA parameter count comparison
model_configs = [
    ('Phi-3-mini (3.8B)', 3_800_000_000, 16,  3_670_016,   0.097),
    ('Mistral-7B',        7_000_000_000, 64,  27_262_976,  0.389),
    ('LLaMA-3-70B',      70_000_000_000, 128, 167_772_160, 0.240),
]
print(f'{"Model":<20} {"Base Params":<15} {"LoRA r":<8} {"LoRA Params":<15} {"% Trainable"}')
print('-' * 70)
for name, base, r, lora, pct in model_configs:
    print(f'{name:<20} {base/1e9:.1f}B{"":<9} {r:<8} {lora/1e6:.1f}M{"":<9} {pct:.3f}%')

## 4. QLoRA — 4-bit Quantization + LoRA

In [ ]:
from transformers import BitsAndBytesConfig
import torch

def setup_qlora(
    model_name: str = 'meta-llama/Llama-3-8b-hf',
) -> tuple:
    """QLoRA: Quantize base model to 4-bit, train LoRA adapters in BF16.
    
    Original paper: 'QLoRA: Efficient Finetuning of Quantized LLMs' (Dettmers 2023)
    Key innovation: NF4 (NormalFloat4) quantization + double quantization
    Enables: fine-tune 65B model on single 48GB GPU!
    """
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type='nf4',           # NormalFloat4 — better than fp4 for normal distributions
        bnb_4bit_use_double_quant=True,       # quantize the quantization constants too
        bnb_4bit_compute_dtype=torch.bfloat16, # compute in BF16 (dequantize → compute → requantize)
    )
    
    # model = AutoModelForCausalLM.from_pretrained(
    #     model_name,
    #     quantization_config=bnb_config,
    #     device_map='auto',
    # )
    # model = prepare_model_for_kbit_training(model)  # gradient checkpointing
    
    lora_config = LoraConfig(
        r=64, lora_alpha=128,
        target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
        lora_dropout=0.05, bias='none', task_type=TaskType.CAUSAL_LM,
    )
    return bnb_config, lora_config

# VRAM requirements with and without QLoRA
memory_table = [
    ('7B model',  'Full FP16',  14.0, 'Single A100 80GB'),
    ('7B model',  'QLoRA (4b)',  5.5, 'RTX 3090 24GB ✓'),
    ('13B model', 'Full FP16',  26.0, 'Need 2x A100'),
    ('13B model', 'QLoRA (4b)',  9.5, 'RTX 4090 24GB ✓'),
    ('70B model', 'Full FP16', 140.0, 'Need 4x A100 80GB'),
    ('70B model', 'QLoRA (4b)', 42.0, 'Single A100 80GB ✓'),
]
print(f'{"Model":<12} {"Strategy":<15} {"VRAM (GB)":<12} {"Hardware"}')
print('-' * 55)
for model, strategy, vram, hw in memory_table:
    print(f'{model:<12} {strategy:<15} {vram:<12.1f} {hw}')

## 5. DPO — Direct Preference Optimization

In [ ]:
from trl import DPOTrainer, DPOConfig
from datasets import Dataset

# DPO dataset format: prompt + chosen (preferred) + rejected (dispreferred)
dpo_dataset = Dataset.from_dict({
    'prompt': [
        'Explain why vaccines are safe.',
        'How do I lose weight quickly?',
        'Write a function to sort a list in Python.',
    ],
    'chosen': [
        'Vaccines undergo rigorous clinical trials with thousands of participants before approval. Common side effects are mild and temporary. Serious adverse events are extremely rare...',
        'Sustainable weight loss involves a modest caloric deficit (300-500 cal/day), balanced nutrition, and regular exercise. Aim for 0.5-1kg per week...',
        'Python has built-in sorting: `sorted(lst)` returns a new list, `lst.sort()` sorts in-place. Use `key=` for custom sorting: `sorted(people, key=lambda p: p.age)`.',
    ],
    'rejected': [
        'Vaccines contain chemicals that can be harmful. Many parents report their children got sick after vaccination. You should research this yourself...',
        'Try crash dieting — eat under 500 calories a day. You can lose 5kg in a week this way!',
        'You can use a for loop to sort: iterate through all pairs and swap them if out of order.',
    ],
})

def setup_dpo_training(sft_model_path: str):
    """DPO: alignment without a reward model.
    
    RLHF requires: SFT model → Reward model → PPO training (complex!)
    DPO directly:  SFT model + preference data → aligned model (simple!)
    
    Loss: L_DPO = -E[log σ(β(log π(y_w|x)/π_ref(y_w|x) - log π(y_l|x)/π_ref(y_l|x)))]
    Intuition: increase probability of chosen, decrease probability of rejected.
    """
    dpo_config = DPOConfig(
        output_dir='./dpo-output',
        beta=0.1,              # KL penalty strength: higher = stay closer to reference
        num_train_epochs=1,
        per_device_train_batch_size=2,
        learning_rate=5e-5,
        bf16=True,
        loss_type='sigmoid',   # original DPO; alternatives: 'ipo', 'hinge'
    )
    return dpo_config

print('DPO vs RLHF comparison:')
comparison = [
    ('Data needed',   'Preference pairs (chosen/rejected)', 'Reward model training data'),
    ('Steps',         '1 (DPO training)',                   '3 (SFT → RM → PPO)'),
    ('Complexity',    'Simple — standard gradient descent', 'Complex — RL instability'),
    ('Memory',        '2x model (model + reference)',       '3x+ model (model + RM + ref)'),
    ('Performance',   'Competitive with RLHF',             'Slightly better in some evals'),
]
print(f'{"Aspect":<15} {"DPO":<42} {"RLHF"}')
print('-' * 85)
for aspect, dpo, rlhf in comparison:
    print(f'{aspect:<15} {dpo:<42} {rlhf}')

## Exercises

1. **LoRA rank ablation:** Fine-tune a small model (Phi-3-mini) with LoRA ranks r={4, 8, 16, 32, 64}. Plot val loss vs number of trainable parameters. Find the knee of the curve.

2. **DPO dataset pipeline:** Build a pipeline that: (1) generates 100 instruction-response pairs, (2) sends each to two models (gpt-4o-mini and gpt-3.5-turbo), (3) uses gpt-4o to judge which response is better, (4) saves the preference dataset in DPO format.

3. **Model merging:** Merge two LoRA adapters fine-tuned on different tasks using SLERP (Spherical Linear Interpolation). Evaluate the merged model on both tasks and compare to each individual adapter.